In [ ]:
import os, io
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import zipfile
import gzip
import requests

import tensorflow as tf

from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

from tensorflow import keras
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.losses import CategoricalCrossentropy
from tensorflow.keras.layers import Conv2D, ReLU, MaxPooling2D, UpSampling2D, Dropout, BatchNormalization, Flatten, Dense, Conv2DTranspose, GlobalAveragePooling2D, DepthwiseConv2D
from tensorflow.keras.callbacks import ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [ ]:
def get_dataset(train_generator, batch_size=32, val_split=0.2, random_state=42, option='MNIST'):
    data = ''
    if(option == 'MNIST'):
        data = tf.keras.datasets.mnist
    elif(option == 'fashionMNIST'):
        data = tf.keras.datasets.fashion_mnist
    elif(option == 'cifar10'):
        data = tf.keras.datasets.cifar10
        
    (x_train, y_train), (x_test, y_test) = data.load_data()

    if option != 'cifar10':
        x_train = x_train.reshape(-1, 28, 28, 1).astype("float32") / 255.0
        x_test = x_test.reshape(-1, 28, 28, 1).astype("float32") / 255.0
    else:
        x_train = x_train.astype("float32") / 255.0
        x_test = x_test.astype("float32") / 255.0
    #Split data to train and valitation
    x_train, x_val, y_train, y_val = train_test_split(x_train, y_train, test_size = val_split, random_state=random_state)

    num_classes = 10  # MNIST has 10 classes (digits 0-9)
    y_train = to_categorical(y_train, num_classes)
    y_val = to_categorical(y_val, num_classes)
    y_test = to_categorical(y_test, num_classes)

    #Use default ImageDataGenerator for inmutable images
    val_test_generator = ImageDataGenerator()
    train_gen = train_generator.flow(x_train, y_train, batch_size=batch_size)
    val_gen = val_test_generator.flow(x_val, y_val, batch_size=batch_size)
    test_gen = val_test_generator.flow(x_test, y_test, batch_size=batch_size, shuffle=False)

    return train_gen, val_gen, test_gen, y_test

In [ ]:
train_generator = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.15,
    height_shift_range=0.15,
    shear_range=0.2,
    zoom_range=0.2)

train_data_mnist, val_data_mnist, test_data_mnist, y_true_mnist = get_dataset(train_generator, batch_size=64, val_split=0.2, random_state=42, option='MNIST')


In [ ]:
def create_model(option='MNIST'):
    model = keras.Sequential()
    model.add(Conv2D(16, (5, 5), strides=(1, 1), activation=None, padding='same', input_shape=(28, 28, 1)))
    model.add(BatchNormalization())
    model.add(ReLU())

    model.add(DepthwiseConv2D((3, 3), padding='same'))
    model.add(BatchNormalization())
    model.add(ReLU())
    
    model.add(DepthwiseConv2D((3, 3), padding='same'))
    model.add(BatchNormalization())
    model.add(ReLU())
    
    model.add(Conv2D(32, (3, 3), padding='same', activation=None))
    model.add(BatchNormalization())
    model.add(ReLU())
    model.add(MaxPooling2D((2, 2)))
    model.add(Dropout(0.2))
    
    model.add(DepthwiseConv2D((3, 3), padding='same'))
    model.add(BatchNormalization())
    model.add(ReLU())
    
    model.add(DepthwiseConv2D((3, 3), padding='same'))
    model.add(BatchNormalization())
    model.add(ReLU())

    model.add(Conv2D(64, (3, 3), padding='same', activation=None))
    model.add(BatchNormalization())
    model.add(ReLU())
    model.add(MaxPooling2D((2, 2)))
    model.add(Dropout(0.2))

    model.add(DepthwiseConv2D((3, 3), padding='same'))
    model.add(BatchNormalization())
    model.add(ReLU())
    
    model.add(DepthwiseConv2D((3, 3), padding='same'))
    model.add(BatchNormalization())
    model.add(ReLU())

    model.add(Conv2D(64, (3, 3), padding='same', activation=None))
    model.add(BatchNormalization())
    model.add(ReLU())
    model.add(MaxPooling2D((2, 2)))
    model.add(Dropout(0.2))

    model.add(DepthwiseConv2D((3, 3), padding='same'))
    model.add(BatchNormalization())
    model.add(ReLU())
    
    model.add(DepthwiseConv2D((3, 3), padding='same'))
    model.add(BatchNormalization())
    model.add(ReLU())

    model.add(Conv2D(64, (3, 3), padding='same', activation=None))
    model.add(BatchNormalization())
    model.add(ReLU())
    model.add(MaxPooling2D((2, 2)))
    model.add(Dropout(0.2))

    model.add(DepthwiseConv2D((3, 3), padding='same'))
    model.add(BatchNormalization())
    model.add(ReLU())
    
    model.add(DepthwiseConv2D((3, 3), padding='same'))
    model.add(BatchNormalization())
    model.add(ReLU())

    model.add(GlobalAveragePooling2D())
    model.add(Dropout(0.3))
    model.add(Dense(10, activation='softmax')) 

    optimizer = Adam(learning_rate=2e-3)
    loss_fn = CategoricalCrossentropy(label_smoothing=0.1)
    model.compile(optimizer=optimizer,
                  loss=loss_fn,
                  metrics=['accuracy', 'Precision', 'Recall'])
    model.summary()

    return model

model_mnist = create_model(option='MNIST')

In [ ]:
lr_scheduler_mnist = ReduceLROnPlateau(monitor='loss', patience=3, factor=0.5, min_lr=5e-5)

In [ ]:
history_mnist = model_mnist.fit(train_data_mnist,
         validation_data=val_data_mnist,
         epochs=50,
         batch_size=64,
         shuffle=True
         ,callbacks=[lr_scheduler_mnist])